# 01 — Corpus Preparation

This notebook prepares Italian parliamentary speeches for the analysis
of health-related discourse and linguistic differences between parties.

Starting from the original ItaParlCorpus data, it:

1. Selects speeches from legislature XVIII within 2018–2022.
2. Removes chair interventions and cleans transcription artifacts.
3. Tokenizes and lemmatizes Italian text for lexical analysis.
4. Flags records eligible for topic modelling.
5. Identifies PD, FdI, M5S, Forza Italia and Lega for the subsequent comparison.

The original text and metadata are preserved. All eligible speeches,
including those outside the five selected parties, remain available
for topic modelling. Health-related speeches are identified in notebook 02.

**Input:** [`camera_2006-2022.csv`](https://dataverse.harvard.edu/file.xhtml?fileId=10876411&version=3.0), from ItaParlCorpus v3.0, Harvard Dataverse.

**Dataset DOI:** https://doi.org/10.7910/DVN/KUARWD

**Output:** `data/processed/corpus_prepared.parquet`

Run the cells in order. Files created under `/content` are temporary:
download the final output before ending the Colab session.

## 1. Set Up the Workspace and Download the Source Data

Create the directories used by this notebook and download the original
CSV from Harvard Dataverse. Reuse the local file if it is already present.

In [1]:
from pathlib import Path

import requests

PROJECT_DIR = Path("/content/italian-parliament-health-discourse")
RAW_PATH = PROJECT_DIR / "data/raw/camera_2006-2022.csv"

DOWNLOAD_URL = (
    "https://dataverse.harvard.edu/api/access/datafile/10876411"
)

for folder in ["data/raw", "data/processed", "results/tables"]:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

if RAW_PATH.is_file():
    print(f"Using existing file: {RAW_PATH.name}")
else:
    temporary_path = RAW_PATH.with_suffix(".csv.part")

    with requests.get(
        DOWNLOAD_URL,
        stream=True,
        timeout=(30, 120),
    ) as response:
        response.raise_for_status()

        content_type = response.headers.get("Content-Type", "").lower()
        if "html" in content_type or "json" in content_type:
            raise RuntimeError(
                "Dataverse returned a webpage or API message instead of the CSV."
            )

        with temporary_path.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)

    temporary_path.replace(RAW_PATH)
    print(f"Downloaded: {RAW_PATH.name}")

print(f"File size: {RAW_PATH.stat().st_size / 1024**2:.1f} MiB")

Downloaded: camera_2006-2022.csv
File size: 653.4 MiB


## 2. Check the Source Structure

Read five records to verify the required columns without loading the full
corpus. Read all fields as strings to preserve identifiers such as
`20060428.0001`; dates are converted explicitly during extraction.

In [2]:
import pandas as pd

EXPECTED_COLUMNS = [
    "date", "doc_id", "row_id", "legislature", "speaker",
    "pageid_wiki", "party_name", "party_family",
    "party_id_parlgov", "party_id_itaparl",
    "chair", "cabinet", "text", "year",
]

preview = pd.read_csv(
    RAW_PATH,
    dtype="string",
    encoding="utf-8-sig",
    nrows=5,
)

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(preview.columns))

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print(f"Required columns found: {len(EXPECTED_COLUMNS)}")

display(
    preview[
        ["date", "row_id", "speaker", "party_name", "chair", "cabinet"]
    ]
)

Required columns found: 14


,date,row_id,speaker,party_name,chair,cabinet
0,2006-04-28,20060428,PRESIDENTE,Chair,TRUE,FALSE
1,2006-04-28,20060428.0001,ELIO VITO,Forza Italia – Il Popolo della Libertà,FALSE,FALSE
2,2006-04-28,20060428.0002,ENRICO BOSELLI,Rosa nel Pugno,FALSE,FALSE
3,2006-04-28,20060428.0003,PRESIDENTE,Chair,TRUE,FALSE
4,2006-04-28,20060428.0004,ENRICO BOSELLI,Rosa nel Pugno,FALSE,FALSE


## 3. Extract the 2018–2022 Corpus

Read the source CSV in chunks and retain records dated from January 2018
through December 2022. Preserve the original fields and record positions.

Collect records with missing or invalid dates separately to check whether
they could affect the study sample.

In [3]:
START_DATE = pd.Timestamp("2018-01-01")
END_DATE = pd.Timestamp("2023-01-01")  # Exclusive

selected_chunks = []
invalid_chunks = []
total_rows = 0

with pd.read_csv(
    RAW_PATH,
    dtype="string",
    encoding="utf-8-sig",
    chunksize=20_000,
) as reader:
    for chunk in reader:
        total_rows += len(chunk)

        dates = pd.to_datetime(
            chunk["date"],
            format="%Y-%m-%d",
            errors="coerce",
        )

        if dates.isna().any():
            invalid_chunks.append(
                chunk.loc[
                    dates.isna(),
                    ["date", "year", "legislature", "row_id"],
                ].copy()
            )

        in_period = dates.ge(START_DATE) & dates.lt(END_DATE)
        selected = chunk.loc[in_period].copy()

        if not selected.empty:
            selected["source_record"] = selected.index + 1
            selected["date_parsed"] = dates.loc[in_period]
            selected_chunks.append(selected)

if not selected_chunks:
    raise ValueError("No records found in the requested period.")

corpus = pd.concat(selected_chunks, ignore_index=True)

unresolved_dates = (
    pd.concat(invalid_chunks, ignore_index=True)
    if invalid_chunks
    else pd.DataFrame(columns=["date", "year", "legislature", "row_id"])
)

del selected_chunks, invalid_chunks

print(f"Original records read: {total_rows:,}")
print(f"Records selected for 2018–2022: {len(corpus):,}")
print(
    f"Selected dates: {corpus['date_parsed'].min().date()} "
    f"to {corpus['date_parsed'].max().date()}"
)

print("\nSelected records by year:")
display(corpus["date_parsed"].dt.year.value_counts().sort_index())

print(f"\nMissing or invalid dates: {len(unresolved_dates):,}")
display(
    unresolved_dates["legislature"]
    .value_counts(dropna=False)
    .rename("records")
)

Original records read: 515,481
Records selected for 2018–2022: 121,936
Selected dates: 2018-01-09 to 2022-09-28

Selected records by year:


,count
date_parsed,
2018,22623
2019,33562
2020,23483
2021,26422
2022,15846



Missing or invalid dates: 259


,records
legislature,
15,259


## 4. Define the Study Sample

Retain legislature XVIII and exclude interventions marked as chair
contributions. Preserve the original metadata and add a quarterly variable.

Check essential fields and unique record identifiers. Missing party-family
values do not cause exclusion. Text-quality filters are applied later.

In [4]:
legislature_18 = corpus.loc[corpus["legislature"].eq("18")].copy()

chair_flag = legislature_18["chair"].str.strip().str.upper()

if not chair_flag.isin(["TRUE", "FALSE"]).all():
    raise ValueError("Missing or unexpected values in the chair indicator.")

study = legislature_18.loc[chair_flag.eq("FALSE")].copy()

study["quarter"] = (
    study["date_parsed"].dt.to_period("Q").astype(str).str.lower()
)

for column in ["row_id", "speaker", "text"]:
    missing = study[column].fillna("").str.strip().eq("")
    if missing.any():
        raise ValueError(
            f"{missing.sum()} records have missing or blank {column}."
        )

if study["row_id"].duplicated().any():
    raise ValueError("Repeated row IDs in the study sample.")

sample_flow = pd.DataFrame({
    "step": [
        "All extracted records, 2018–2022",
        "Legislature XVIII",
        "After excluding chair interventions",
    ],
    "records": [
        len(corpus),
        len(legislature_18),
        len(study),
    ],
})

sample_flow.to_csv(
    PROJECT_DIR / "results/tables/sample_flow.csv",
    index=False,
)

display(sample_flow)

print(
    f"Study period: {study['date_parsed'].min().date()} "
    f"to {study['date_parsed'].max().date()}"
)

,step,records
0,"All extracted records, 2018–2022",121936
1,Legislature XVIII,121862
2,After excluding chair interventions,61985


Study period: 2018-03-29 to 2022-09-28


## 5. Clean Speech Text

Create `text_clean` while preserving the original `text` column.
Remove the leading speaker header and parenthetical notes beginning
with recognized stage directions.

Preserve punctuation, numbers, accents, negations and other parenthetical
content. Count whitespace-separated words for the later length filter;
no records are excluded at this stage.

In [5]:
import re
import unicodedata

STAGE_DIRECTION = re.compile(
    r"\(\s*(?:"
    r"applaus\w*|generali\s+applaus\w*|viv\w*\s+applaus\w*|"
    r"comment\w*|protest\w*|interruzion\w*|"
    r"ilarità|si\s+ride|segue\s+la\s+votazione"
    r")\b[^()]*\)",
    flags=re.IGNORECASE,
)


def clean_speech(text, speaker):
    """Remove speaker headers and recognized transcription notes."""
    cleaned = unicodedata.normalize("NFC", str(text))
    name = unicodedata.normalize("NFC", str(speaker)).strip()

    if name:
        flexible_name = r"\s+".join(
            re.escape(part) for part in name.split()
        )
        header = (
            r"^\s*" + flexible_name
            + r"(?:\s*\([^()\n]*\))?"
            + r"\s*[.,:;–—-]+\s*"
        )
        cleaned = re.sub(
            header, "", cleaned, count=1, flags=re.IGNORECASE
        )

    cleaned = STAGE_DIRECTION.sub(" ", cleaned)
    return re.sub(r"\s+", " ", cleaned).strip()


study["text_clean"] = [
    clean_speech(text, speaker)
    for text, speaker in zip(study["text"], study["speaker"])
]

study["n_words_clean"] = study["text_clean"].str.split().str.len()

print(f"Records processed: {len(study):,}")
print(f"Empty cleaned texts: {study['text_clean'].eq('').sum():,}")

examples = study.sample(2, random_state=42)[
    ["row_id", "text", "text_clean"]
].copy()

for column in ["text", "text_clean"]:
    examples[column] = examples[column].str.slice(0, 300)

with pd.option_context("display.max_colwidth", 300):
    display(examples)

Records processed: 61,985
Empty cleaned texts: 3


,row_id,text,text_clean
15285,20181106.0026,FRANCESCO FORCINITI (M5S). Chiedo di parlare.,Chiedo di parlare.
90283,20210512.0208,MASSIMO UNGARO (IV). Anche noi annunciamo il voto favorevole a nome di Italia Viva (Applausi dei deputati del gruppo Italia Viva).,Anche noi annunciamo il voto favorevole a nome di Italia Viva .


## 6. Tokenize and Lemmatize Italian Speeches

Use spaCy to tokenize and lemmatize `text_clean`. Retain alphabetic
lemmas longer than one character and remove stopwords, preserving
the selected negations: non, né, mai and senza.

Save the retained lemmas in `text_lexical` for LDA and lexical analysis.
Keep `text_clean` unchanged for semantic embeddings.

Pin the spaCy and Italian language-model versions for reproducible text processing.

In [6]:
%pip install -q "spacy==3.8.16" tqdm
%pip install -q "https://github.com/explosion/spacy-models/releases/download/it_core_news_sm-3.8.0/it_core_news_sm-3.8.0-py3-none-any.whl"

In [7]:
import spacy
from tqdm.auto import tqdm

nlp = spacy.load("it_core_news_sm", disable=["parser", "ner"])

PRESERVED_NEGATIONS = {"non", "né", "mai", "senza"}
STOP_WORDS = {
    word.casefold() for word in nlp.Defaults.stop_words
} - PRESERVED_NEGATIONS


def extract_lemmas(doc):
    """Select alphabetic lemmas, excluding stopwords and one-letter items."""
    lemmas = []

    for token in doc:
        lemma = token.lemma_.strip().casefold()

        if (
            token.is_alpha
            and len(lemma) > 1
            and token.text.casefold() not in STOP_WORDS
            and lemma not in STOP_WORDS
        ):
            lemmas.append(lemma)

    return lemmas


lexical_texts = []
token_counts = []

documents = nlp.pipe(
    study["text_clean"],
    batch_size=64,
    n_process=1,
)

for doc in tqdm(
    documents,
    total=len(study),
    desc="Italian lemmatization",
):
    lemmas = extract_lemmas(doc)
    lexical_texts.append(" ".join(lemmas))
    token_counts.append(len(lemmas))

study["text_lexical"] = lexical_texts
study["n_lexical_tokens"] = token_counts

print(f"Processed records: {len(study):,}")
print(
    "Records with no retained lemmas:",
    study["n_lexical_tokens"].eq(0).sum(),
)

examples = study.sample(2, random_state=42)[
    ["row_id", "text_clean", "text_lexical"]
].copy()

for column in ["text_clean", "text_lexical"]:
    examples[column] = examples[column].str.slice(0, 300)

with pd.option_context("display.max_colwidth", 300):
    display(examples)

Italian lemmatization:   0%|          | 0/61985 [00:00<?, ?it/s]

Processed records: 61,985
Records with no retained lemmas: 160


,row_id,text_clean,text_lexical
15285,20181106.0026,Chiedo di parlare.,chiedere parlare
90283,20210512.0208,Anche noi annunciamo il voto favorevole a nome di Italia Viva .,annuncare voto favorevole nome italia viva


## 7. Define Model Eligibility

Flag suspected transcription headings, empty cleaned texts, speeches with
fewer than 20 whitespace-separated words, and speeches with fewer than
5 retained lexical tokens.

A record is eligible only if none of these flags applies. These are
operational quality filters, not indicators of health-related content.

Preserve all records and exclusion reasons. Use the same eligible
speeches for BERTopic and LDA.

In [8]:
MIN_WORDS = 20
MIN_LEXICAL_TOKENS = 5

study["flag_heading"] = study["speaker"].str.contains(
    r"^\s*(?:PRESIDENZA\b|PRESIDENTE\b|VICEPRESIDENTE\b|ANSFISA$)",
    case=False,
    regex=True,
    na=False,
)

study["flag_empty_text"] = study["text_clean"].eq("")
study["flag_short_text"] = study["n_words_clean"].lt(MIN_WORDS)
study["flag_few_lemmas"] = (
    study["n_lexical_tokens"].lt(MIN_LEXICAL_TOKENS)
)

EXCLUSION_FLAGS = [
    "flag_heading",
    "flag_empty_text",
    "flag_short_text",
    "flag_few_lemmas",
]

study["eligible_for_models"] = ~study[EXCLUSION_FLAGS].any(axis=1)

study["exclusion_reason"] = [
    "; ".join(
        flag.removeprefix("flag_")
        for flag, excluded in zip(EXCLUSION_FLAGS, values)
        if excluded
    )
    for values in study[EXCLUSION_FLAGS].itertuples(
        index=False, name=None
    )
]

print(f"Starting records: {len(study):,}")
print(f"Eligible for both models: {study['eligible_for_models'].sum():,}")
print(f"Excluded records: {(~study['eligible_for_models']).sum():,}")

print("\nExclusion flags — counts may overlap:")
display(study[EXCLUSION_FLAGS].sum().rename("records"))

Starting records: 61,985
Eligible for both models: 49,629
Excluded records: 12,356

Exclusion flags — counts may overlap:


,records
flag_heading,39
flag_empty_text,3
flag_short_text,12340
flag_few_lemmas,6494


## 8. Identify the Five Parties

Use the original `party_name` labels to identify PD, FdI, M5S,
Forza Italia and Lega for the subsequent comparison.

Aggregate labels such as Centrosinistra and Centrodestra are not assigned
to parties or ideological blocs. All records remain in the preparation
table, and all eligible speeches remain available for topic modelling.

Party assignments follow the source metadata; they do not reconstruct
each speaker's affiliation history.

In [9]:
PARTY_NAMES = {
    "Partito Democratico": "PD",
    "Fratelli d'Italia": "FdI",
    "Movimento 5 Stelle": "M5S",
    "Forza Italia – Il Popolo della Libertà": "FI",
    "Lega": "Lega",
}

study["analysis_party"] = study["party_name"].map(PARTY_NAMES)
study["in_five_party_comparison"] = study["analysis_party"].notna()

party_counts = (
    study.loc[
        study["eligible_for_models"]
        & study["in_five_party_comparison"]
    ]
    .groupby("analysis_party")
    .size()
    .rename("records")
    .sort_values(ascending=False)
)

display(party_counts)

print(f"Eligible speeches in the five-party comparison: {party_counts.sum():,}")
print(
    "Other eligible speeches retained for topic modelling:",
    f"{(study['eligible_for_models'] & ~study['in_five_party_comparison']).sum():,}",
)

,records
analysis_party,
PD,10150
FdI,9697
M5S,7169
FI,6643
Lega,5413


Eligible speeches in the five-party comparison: 39,072
Other eligible speeches retained for topic modelling: 10,557


## 9. Export the Prepared Corpus

Save the complete preparation table as a single compressed Parquet file,
including original metadata, both text representations, quality flags,
exclusion reasons and five-party indicators.

Preserve source-record order. The exported file can be uploaded to GitHub
using Git LFS.

In [10]:
OUTPUT_PATH = PROJECT_DIR / "data/processed/corpus_prepared.parquet"

prepared = study.sort_values("source_record").reset_index(drop=True)

prepared.to_parquet(
    OUTPUT_PATH,
    index=False,
    compression="zstd",
)

print(f"Exported records: {len(prepared):,}")
print(f"Eligible records: {prepared['eligible_for_models'].sum():,}")
print(f"Columns: {len(prepared.columns)}")
print(f"Saved: {OUTPUT_PATH.name}")
print(f"File size: {OUTPUT_PATH.stat().st_size / 1024**2:.1f} MiB")

Exported records: 61,985
Eligible records: 49,629
Columns: 29
Saved: corpus_prepared.parquet
File size: 122.6 MiB


In [ ]:
from google.colab import files

files.download(str(OUTPUT_PATH))